# Flaky-Test Root-Cause Diagnosis: Evidence Requirements Pilot

**Purpose.** This notebook contains a small pilot study investigating which evidence sources are useful and minimally sufficient for flaky-test root-cause diagnosis.

## Research question

> **Which evidence types are minimally sufficient for reliable flaky-test root-cause diagnosis, and how does evidence sufficiency vary across root-cause categories?**

### Pilot design
- **Dataset:** RustFT material associated with FTW
- **Pilot:** 15 selected cases
- **Root-cause categories:** 9
- **Automated baseline:** `facebook/bart-large-mnli` zero-shot classification
- **Evidence levels:** issue description, failure evidence, runtime evidence, source/execution-path evidence, and retrieved external artifacts
- **Two complementary analyses:** controlled evidence ablation + human minimum-sufficient-evidence annotation

> **Important:** The automated results are a small pilot and should be interpreted as hypothesis-generating observations, not as evidence about modern LLMs in general.

## Notebook structure

1. Load RustFT and define the pilot cases
2. Define the evidence taxonomy
3. Build clean evidence segments and cumulative evidence packages
4. Run the BART zero-shot evidence-ablation experiment
5. Examine genuine external artifacts separately
6. Analyze minimum sufficient evidence
7. Analyze root-cause × evidence patterns
8. Define the annotation protocol and blind second pass
9. Summarize findings, limitations, and next steps

## Reproducibility note

The notebook is organized for **GitHub review and Kaggle execution**. The RustFT CSV is not bundled here; the Kaggle input path is used when the notebook is run in Kaggle.

The GitHub version intentionally does **not** include credentials or API tokens. GitHub issue retrieval in the exploratory portion uses public API endpoints.

In [1]:
import pandas as pd
import numpy as np
import re
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Kaggle input path
DATA_PATH = "/kaggle/input/datasets/audityghosh/rustft-csv/issues-ftw25-artifact.csv"

print("Data path:", DATA_PATH)

Data path: /kaggle/input/datasets/audityghosh/rustft-csv/issues-ftw25-artifact.csv


In [2]:
# Load the original RustFT dataset

import pandas as pd

csv_path = DATA_PATH

rustft = pd.read_csv(csv_path)

print("Original dataset shape:", rustft.shape)

print("\nColumns:")
print(rustft.columns.tolist())

Original dataset shape: (87, 12)

Columns:
['id', 'user_name', 'repo_name', 'stars', 'repo_url', 'issue_url', 'linked_pr_url', 'root_cause_category', 'root_cause_subcategory', 'fix_category', 'fix_effectiveness', 'change_scope']


## 1. Pilot cases

The pilot is deliberately small so that each case can be inspected carefully and its evidence can be segmented manually before scaling the study.

In [3]:
pilot_issue_data = [
    {
        "case": "nearcore",
        "ground_truth": "Randomness",
        "issue_body": """https://github.com/near/nearcore/pull/10015 introduced a flaky test test_trie_consistency_random, fails about 2% of the time; will fix at first convenience."""
    },

    {
        "case": "reth",
        "ground_truth": "Network",
        "issue_body": """### Describe the bug

https://github.com/paradigmxyz/reth/actions/runs/5405202612/jobs/9820416442?pr=3455#step:8:789

--- STDERR: reth-network::it connect::test_incoming_node_id_blacklist ---
thread 'connect::test_incoming_node_id_blacklist' panicked at 'called `Result::unwrap()` on an `Err` value: HTTPError(reqwest::Error { kind: Request, url: Url { scheme: "http", cannot_be_a_base": false, username: "", password: None, host: Some(127.0.0.1), port: Some(44187), path: "/", query: None, fragment: None }, source: hyper::Error(Connect, ConnectError("tcp connect error", Os { code: 111, kind: ConnectionRefused, message: "Connection refused" }) ) })', crates/net/network/tests/it/connect.rs:325:70"""
    },

    {
        "case": "fedimint_async",
        "ground_truth": "Async Wait",
        "issue_body": """Currently fedimint-cli spend <amount> will give us <amount> if it has those notes on hand, and give more than we ask for if it doesn't. I was just writing a test and this behavior caused my test to be flaky. It would be nice for this reason (and probably in general) to be able to request exact amounts from fedimint-cli spend."""
    },

    {
        "case": "ethersync",
        "ground_truth": "Concurrency",
        "issue_body": """The fuzzer causes some ungraceful shutdown of the tokio tasks, which makes CI fail in a flaky way."""
    },

    {
        "case": "vibranium",
        "ground_truth": "Logic",
        "issue_body": """Current algo to merge default CLI options with custom ones is primitive and flaky. E.g. it relies on sorting which doesn't work well for this use case in the first place."""
    },

    {
        "case": "solana_async",
        "ground_truth": "Async Wait",
        "issue_body": """#### Problem
#5659

#### Proposed Solution
Debug the race condition exposed by test_banking_stage_entryfication, fix it, then revert #5659"""
    },

    {
        "case": "deltachat",
        "ground_truth": "Network",
        "issue_body": """src/smtp.rs::send_smtp_messages logs and ignores SMTP errors when sending MDNs, then returns Ok(). Then in src/scheduler::smtp_loop we reset timeout to None and then log "smtp has no messages to retry" and wait for interrupt forever.

It is not true that SMTP has no messages to retry in this case.

The worst part of this is that during high load postfix returns temporary error "421 4.4.2 Error: timeout exceeded". Then tests time out in CI because they never retry sending."""
    },

    {
        "case": "fedimint_logic",
        "ground_truth": "Logic",
        "issue_body": """I have twice seen this error when running scripts/cli-test.sh:

Error: Peg-out address received 0.000005 BTC, expected 0.00000500

The two numbers are equal, but padded differently with zeros."""
    },

    {
        "case": "parsec-cloud",
        "ground_truth": "Hard to classify",
        "issue_body": """The hypothesis test during the action [2887918745](...) failed because of that. The log trace is contained in macos-hypothesis-test.zip(...)"""
    },

    {
        "case": "relay",
        "ground_truth": "Time",
        "issue_body": """### Flakiness Type
Assertion failure

### Name of Test
extractors::start_time::tests::start_time_from_timestamp

### Link to Test Run
https://github.com/getsentry/relay/actions/runs/8050926586/job/21987677539

### Details
Seems like an issue between the generation of the now timestamp and the system_time timestamp. We're likely crossing from second n to second n+1 between calls.

assertion `left == right` failed
left: 9
right: 10

test result: FAILED; 225 passed; 1 failed"""
    },

    {
        "case": "solana_randomness",
        "ground_truth": "Randomness",
        "issue_body": """#### Problem
#18278 ignores 2 local-cluster tests failing consistently in CI

#### Proposed Solution
Identify root cause of test failures, fix, and re-enable them"""
    },

    {
        "case": "diem",
        "ground_truth": "I/O",
        "issue_body": """# Bug

I have seen a few PRs fail with no code changes in storage::command_adapter::tests::test_save_list_metadata_files.

Two runs are:
https://circleci.com/gh/libra/libra/167227
https://circleci.com/gh/libra/libra/166268"""
    },

    {
        "case": "rust-lightning",
        "ground_truth": "Concurrency",
        "issue_body": """Looks like the new fuzz_threaded_connections is flaky, it can hit an unwrap on the first read_event, which shouldn't fail, but CI managed to make it. This isn't an immediate correctness concern, as that erroring isn't a problem, just surprising."""
    },

    {
        "case": "databend",
        "ground_truth": "Unordered data",
        "issue_body": """This looks like a flaky test. I will fix it.

05_0001_set_var: [ FAIL ] - result differs with:

--- result
+++ stdout

-x x
 timezone America/Los_Angeles
 x x
+x x
 timezone Asia/Shanghai"""
    },

    {
        "case": "webrender",
        "ground_truth": "I/O",
        "issue_body": """The text/split-batch.yaml fails with 25 pixel difference, and the image/tile-repeat-prim-or-decompose.yaml crashes, because we run out of descriptors.

The fail was introduced by a change involving max_image_array_layers. This only affects Vulkan backend."""
    }
]

pilot_raw = pd.DataFrame(pilot_issue_data)

print("Cases:", len(pilot_raw))
display(pilot_raw[["case", "ground_truth"]])

Cases: 15


,case,ground_truth
0,nearcore,Randomness
1,reth,Network
2,fedimint_async,Async Wait
3,ethersync,Concurrency
4,vibranium,Logic
5,solana_async,Async Wait
6,deltachat,Network
7,fedimint_logic,Logic
8,parsec-cloud,Hard to classify
9,relay,Time


## 2. Evidence taxonomy

The evidence is separated so that the experiment can test **which information changes diagnosis**, rather than simply giving the model every available artifact.

- **E0:** Issue description / author's explanation
- **E1:** Failure evidence — assertions, expected-vs-actual output, exceptions
- **E2:** Runtime evidence — runtime errors, stack traces, timing behavior, network errors, task behavior
- **E3:** Source/execution-path evidence — relevant code, algorithms, causal execution paths
- **E4a:** Retrieved CI/runtime artifacts
- **E4b:** Retrieved attached diagnostic artifacts
- **E4c:** Retrieved PR/commit artifacts

A URL pointing to a CI run or PR is **not** counted as evidence unless the actual artifact content is retrieved and inspected.

In [4]:
# Pilot ground-truth and minimum-evidence annotations

pilot_evidence = pd.DataFrame({
    "case": [
        "nearcore", "reth", "fedimint_async", "ethersync", "vibranium",
        "solana_async", "deltachat", "fedimint_logic", "parsec-cloud",
        "relay", "solana_randomness", "diem", "rust-lightning",
        "databend", "webrender"
    ],
    "ground_truth": [
        "Randomness", "Network", "Async Wait", "Concurrency", "Logic",
        "Async Wait", "Network", "Logic", "Hard to classify", "Time",
        "Randomness", "I/O", "Concurrency", "Unordered data", "I/O"
    ],
    "minimum_evidence": [
        "E4c", "E2", "UNCERTAIN", "E2", "E3",
        "E3/E4a", "E0", "E0", "E4b", "E0",
        "E4a", "E4a", "UNCERTAIN", "E0", "E3"
    ],
    "status": [
        "confirmed", "confirmed", "review", "confirmed", "confirmed",
        "review", "confirmed", "confirmed", "confirmed", "confirmed",
        "confirmed", "confirmed", "review", "confirmed", "confirmed"
    ]
})

print("Total pilot cases:", len(pilot_evidence))
print("Confirmed:", (pilot_evidence["status"] == "confirmed").sum())
print("Needs review:", (pilot_evidence["status"] == "review").sum())

display(pilot_evidence)


Total pilot cases: 15
Confirmed: 12
Needs review: 3


,case,ground_truth,minimum_evidence,status
0,nearcore,Randomness,E4c,confirmed
1,reth,Network,E2,confirmed
2,fedimint_async,Async Wait,UNCERTAIN,review
3,ethersync,Concurrency,E2,confirmed
4,vibranium,Logic,E3,confirmed
5,solana_async,Async Wait,E3/E4a,review
6,deltachat,Network,E0,confirmed
7,fedimint_logic,Logic,E0,confirmed
8,parsec-cloud,Hard to classify,E4b,confirmed
9,relay,Time,E0,confirmed


In [5]:
# ============================================================
# MANUAL EVIDENCE SEGMENTATION — 15 PILOT CASES
# ============================================================
#
# E0  = Issue description
# E1  = Failure evidence
# E2  = Runtime evidence
# E3  = Source / execution-path evidence
# E4a = CI / runtime artifact
# E4b = Attached diagnostic artifact
# E4c = PR / commit artifact
#
# IMPORTANT:
# These are manually curated evidence segments.
# We do NOT use keyword extraction here.
# ============================================================

evidence_segments = {

    "nearcore": {
        "E0_issue": "A flaky test named test_trie_consistency_random is reported to fail about 2% of the time.",
        "E1_failure": "The test is reported as failing intermittently.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "A linked PR (near/nearcore/pull/10015) introduced the flaky test."
    },

    "reth": {
        "E0_issue": "The test reth-network::it::connect::test_incoming_node_id_blacklist is reported as flaky.",
        "E1_failure": "The test panicked because Result::unwrap() received an Err value.",
        "E2_runtime": "HTTPError caused by a reqwest/hyper TCP connection error. The connection to 127.0.0.1:44187 was refused.",
        "E3_source": "The failure occurs at crates/net/network/tests/it/connect.rs:325:70.",
        "E4a_CI": "The issue links to a GitHub Actions test run.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "fedimint_async": {
        "E0_issue": "A test is reported as flaky because fedimint-cli spend can return more than the requested amount when the requested notes are unavailable.",
        "E1_failure": "The test exhibits flaky behavior under this unexpected spend-result behavior.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "ethersync": {
        "E0_issue": "A fuzzer causes ungraceful shutdown of Tokio tasks, making CI fail in a flaky way.",
        "E1_failure": "CI fails intermittently.",
        "E2_runtime": "Ungraceful shutdown of Tokio tasks is explicitly reported.",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "The report states that the problem makes CI fail, but no actual CI log is included.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "vibranium": {
        "E0_issue": "The algorithm for merging default CLI options with custom options is described as primitive and flaky.",
        "E1_failure": "The behavior is reported as flaky.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "The issue identifies sorting as part of the algorithm and explains that sorting does not work well for this use case.",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "solana_async": {
        "E0_issue": "test_banking_stage_entryfication is reported as flaky.",
        "E1_failure": "The test is being ignored because of the flaky failure.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "The issue identifies a race condition exposed by test_banking_stage_entryfication.",
        "E4a_CI": "The issue references an external Buildkite test run containing the failure evidence.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "The linked change ignores test_banking_stage_entryfication."
    },

    "deltachat": {
        "E0_issue": "SMTP errors during MDN sending are logged and ignored, causing the scheduler to believe there are no messages to retry.",
        "E1_failure": "Tests eventually time out in CI.",
        "E2_runtime": "During high load, Postfix can return temporary error 421 4.4.2 timeout exceeded; the scheduler resets the timeout and waits indefinitely instead of retrying.",
        "E3_source": "The issue identifies send_smtp_messages and scheduler::smtp_loop, including the relevant retry/control-flow behavior.",
        "E4a_CI": "The issue states that tests time out in CI.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "fedimint_logic": {
        "E0_issue": "A test reports an error where the received BTC amount and expected BTC amount are numerically equal but formatted differently.",
        "E1_failure": "Received: 0.000005 BTC; expected: 0.00000500 BTC.",
        "E2_runtime": "The error occurs when running scripts/cli-test.sh.",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "parsec-cloud": {
        "E0_issue": "A hypothesis test failed during a GitHub Actions run.",
        "E1_failure": "The issue only states that the hypothesis test failed.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "The issue references GitHub Actions run 2887918745.",
        "E4b_attachment": "A diagnostic log trace is contained in macos-hypothesis-test.zip.",
        "E4c_PR": "NOT_PRESENT"
    },

    "relay": {
        "E0_issue": "The start_time_from_timestamp test is flaky because the generation of the now timestamp and system_time timestamp may cross a second boundary.",
        "E1_failure": "Assertion failed: left = 9, right = 10. The test result was 225 passed and 1 failed.",
        "E2_runtime": "The failure occurs when execution crosses from second n to second n+1 between timestamp calls.",
        "E3_source": "The affected test is extractors::start_time::tests::start_time_from_timestamp.",
        "E4a_CI": "A GitHub Actions test-run link is provided.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "solana_randomness": {
        "E0_issue": "Two local-cluster tests are reported as failing consistently in CI and are ignored.",
        "E1_failure": "Two tests fail in CI.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "The issue identifies the affected local-cluster tests but does not provide their execution mechanism.",
        "E4a_CI": "The evidence of failure is located in CI, but the actual CI output is not included in the issue.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "The change ignores the two failing tests."
    },

    "diem": {
        "E0_issue": "The storage::command_adapter::tests::test_save_list_metadata_files test causes several PRs to fail despite no code changes.",
        "E1_failure": "The PRs fail, but the actual failure output is not included.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "The affected test is identified, but its execution mechanism is not described.",
        "E4a_CI": "Two CircleCI test runs are linked as evidence.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "The report mentions PR failures but provides no specific patch or commit.",
    },

    "rust-lightning": {
        "E0_issue": "The fuzz_threaded_connections test is reported as flaky.",
        "E1_failure": "The test can hit an unwrap on the first read_event.",
        "E2_runtime": "The first read_event unexpectedly produces an error during execution.",
        "E3_source": "The relevant execution path is the fuzz_threaded_connections test and its first read_event operation, but source code is not provided.",
        "E4a_CI": "The issue says CI managed to trigger the error, but no CI log is included.",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "databend": {
        "E0_issue": "The 05_0001_set_var test is reported as flaky.",
        "E1_failure": "The test output differs: the expected/result and stdout contain different timezone ordering/content.",
        "E2_runtime": "NOT_PRESENT",
        "E3_source": "NOT_PRESENT",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "NOT_PRESENT"
    },

    "webrender": {
        "E0_issue": "text/split-batch.yaml fails and image/tile-repeat-prim-or-decompose.yaml crashes.",
        "E1_failure": "One test has a 25-pixel difference; another crashes because the process runs out of descriptors.",
        "E2_runtime": "The crash occurs because the program runs out of file descriptors.",
        "E3_source": "The failure was introduced by a change involving max_image_array_layers and affects the Vulkan backend.",
        "E4a_CI": "NOT_PRESENT",
        "E4b_attachment": "NOT_PRESENT",
        "E4c_PR": "The issue identifies the change that introduced the failure."
    }
}


# Convert dictionary to dataframe
evidence_df = pd.DataFrame.from_dict(
    evidence_segments,
    orient="index"
).reset_index()

evidence_df = evidence_df.rename(columns={"index": "case"})

# Add ground truth and minimum evidence
evidence_df = evidence_df.merge(
    pilot_evidence[
        ["case", "ground_truth", "minimum_evidence", "status"]
    ],
    on="case",
    how="left"
)

# Reorder
evidence_df = evidence_df[
    [
        "case",
        "ground_truth",
        "minimum_evidence",
        "status",
        "E0_issue",
        "E1_failure",
        "E2_runtime",
        "E3_source",
        "E4a_CI",
        "E4b_attachment",
        "E4c_PR"
    ]
]

print("Evidence packages created:", len(evidence_df))

display(evidence_df)

Evidence packages created: 15


,case,ground_truth,minimum_evidence,status,E0_issue,E1_failure,E2_runtime,E3_source,E4a_CI,E4b_attachment,E4c_PR
0,nearcore,Randomness,E4c,confirmed,A flaky test named test_trie_consistency_rando...,The test is reported as failing intermittently.,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,A linked PR (near/nearcore/pull/10015) introdu...
1,reth,Network,E2,confirmed,The test reth-network::it::connect::test_incom...,The test panicked because Result::unwrap() rec...,HTTPError caused by a reqwest/hyper TCP connec...,The failure occurs at crates/net/network/tests...,The issue links to a GitHub Actions test run.,NOT_PRESENT,NOT_PRESENT
2,fedimint_async,Async Wait,UNCERTAIN,review,A test is reported as flaky because fedimint-c...,The test exhibits flaky behavior under this un...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT
3,ethersync,Concurrency,E2,confirmed,A fuzzer causes ungraceful shutdown of Tokio t...,CI fails intermittently.,Ungraceful shutdown of Tokio tasks is explicit...,NOT_PRESENT,The report states that the problem makes CI fa...,NOT_PRESENT,NOT_PRESENT
4,vibranium,Logic,E3,confirmed,The algorithm for merging default CLI options ...,The behavior is reported as flaky.,NOT_PRESENT,The issue identifies sorting as part of the al...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT
5,solana_async,Async Wait,E3/E4a,review,test_banking_stage_entryfication is reported a...,The test is being ignored because of the flaky...,NOT_PRESENT,The issue identifies a race condition exposed ...,The issue references an external Buildkite tes...,NOT_PRESENT,The linked change ignores test_banking_stage_e...
6,deltachat,Network,E0,confirmed,SMTP errors during MDN sending are logged and ...,Tests eventually time out in CI.,"During high load, Postfix can return temporary...",The issue identifies send_smtp_messages and sc...,The issue states that tests time out in CI.,NOT_PRESENT,NOT_PRESENT
7,fedimint_logic,Logic,E0,confirmed,A test reports an error where the received BTC...,Received: 0.000005 BTC; expected: 0.00000500 BTC.,The error occurs when running scripts/cli-test...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT
8,parsec-cloud,Hard to classify,E4b,confirmed,A hypothesis test failed during a GitHub Actio...,The issue only states that the hypothesis test...,NOT_PRESENT,NOT_PRESENT,The issue references GitHub Actions run 288791...,A diagnostic log trace is contained in macos-h...,NOT_PRESENT
9,relay,Time,E0,confirmed,The start_time_from_timestamp test is flaky be...,"Assertion failed: left = 9, right = 10. The te...",The failure occurs when execution crosses from...,The affected test is extractors::start_time::t...,A GitHub Actions test-run link is provided.,NOT_PRESENT,NOT_PRESENT


In [6]:

# Make a clean copy
evidence_clean = evidence_df.copy()

# ------------------------------------------------------------
# Correct evidence boundaries for the pilot
# ------------------------------------------------------------

# DELTACHAT
# E0 should describe the reported problem.
# The causal mechanism belongs in E2/E3.
evidence_clean.loc[
    evidence_clean["case"] == "deltachat", "E0_issue"
] = (
    "SMTP-related tests are reported to time out under some conditions."
)

evidence_clean.loc[
    evidence_clean["case"] == "deltachat", "E1_failure"
] = (
    "Tests time out in CI."
)

evidence_clean.loc[
    evidence_clean["case"] == "deltachat", "E2_runtime"
] = (
    "During high load, Postfix can return temporary error "
    "'421 4.4.2 timeout exceeded'. SMTP errors occur during message sending."
)

evidence_clean.loc[
    evidence_clean["case"] == "deltachat", "E3_source"
] = (
    "src/smtp.rs::send_smtp_messages logs and ignores SMTP errors "
    "and returns Ok(). In src/scheduler::smtp_loop, timeout is reset "
    "and the scheduler concludes there are no messages to retry."
)

# RELAY
# Keep the basic problem in E0; exact timing mechanism in E2.
evidence_clean.loc[
    evidence_clean["case"] == "relay", "E0_issue"
] = (
    "The start_time_from_timestamp test is reported as flaky."
)

evidence_clean.loc[
    evidence_clean["case"] == "relay", "E1_failure"
] = (
    "Assertion failed: left = 9, right = 10."
)

evidence_clean.loc[
    evidence_clean["case"] == "relay", "E2_runtime"
] = (
    "The now timestamp and system_time timestamp can cross "
    "from second n to second n+1 between calls."
)

# WEBRENDER
# E0 = symptoms; E2 = runtime behavior; E3 = source/change mechanism.
evidence_clean.loc[
    evidence_clean["case"] == "webrender", "E0_issue"
] = (
    "Two WebRender tests are reported as failing/crashing."
)

evidence_clean.loc[
    evidence_clean["case"] == "webrender", "E1_failure"
] = (
    "text/split-batch.yaml has a 25-pixel difference; "
    "image/tile-repeat-prim-or-decompose.yaml crashes."
)

evidence_clean.loc[
    evidence_clean["case"] == "webrender", "E2_runtime"
] = (
    "The crash occurs because the process runs out of file descriptors."
)

evidence_clean.loc[
    evidence_clean["case"] == "webrender", "E3_source"
] = (
    "The failure was introduced by a change involving "
    "max_image_array_layers and affects the Vulkan backend."
)

# VIBRANIUM
# Sorting mechanism is source/algorithm evidence.
evidence_clean.loc[
    evidence_clean["case"] == "vibranium", "E0_issue"
] = (
    "The algorithm for merging default CLI options with custom "
    "options is reported as flaky."
)

evidence_clean.loc[
    evidence_clean["case"] == "vibranium", "E1_failure"
] = (
    "The merging behavior produces incorrect/flaky results."
)

evidence_clean.loc[
    evidence_clean["case"] == "vibranium", "E3_source"
] = (
    "The algorithm relies on sorting, which is identified as "
    "inappropriate for this use case."
)

# FEDIMINT LOGIC
# Formatting mismatch itself is the failure evidence.
evidence_clean.loc[
    evidence_clean["case"] == "fedimint_logic", "E0_issue"
] = (
    "A test reports an error involving the received and expected "
    "Bitcoin amounts."
)

evidence_clean.loc[
    evidence_clean["case"] == "fedimint_logic", "E1_failure"
] = (
    "Received: 0.000005 BTC; expected: 0.00000500 BTC. "
    "The two numerical values are equal but formatted differently."
)

# DATABEND
evidence_clean.loc[
    evidence_clean["case"] == "databend", "E0_issue"
] = (
    "The 05_0001_set_var test is reported as flaky."
)

evidence_clean.loc[
    evidence_clean["case"] == "databend", "E1_failure"
] = (
    "The result differs from stdout, including different ordering "
    "of timezone values."
)

# RETH
evidence_clean.loc[
    evidence_clean["case"] == "reth", "E0_issue"
] = (
    "The test test_incoming_node_id_blacklist is reported as failing."
)

evidence_clean.loc[
    evidence_clean["case"] == "reth", "E1_failure"
] = (
    "The test panicked because Result::unwrap() received an Err value."
)

evidence_clean.loc[
    evidence_clean["case"] == "reth", "E2_runtime"
] = (
    "The Err is an HTTPError caused by a TCP connection error: "
    "ConnectionRefused when connecting to 127.0.0.1:44187."
)

# Show the cleaned version
display(
    evidence_clean[
        [
            "case",
            "ground_truth",
            "minimum_evidence",
            "E0_issue",
            "E1_failure",
            "E2_runtime",
            "E3_source"
        ]
    ]
)

,case,ground_truth,minimum_evidence,E0_issue,E1_failure,E2_runtime,E3_source
0,nearcore,Randomness,E4c,A flaky test named test_trie_consistency_rando...,The test is reported as failing intermittently.,NOT_PRESENT,NOT_PRESENT
1,reth,Network,E2,The test test_incoming_node_id_blacklist is re...,The test panicked because Result::unwrap() rec...,The Err is an HTTPError caused by a TCP connec...,The failure occurs at crates/net/network/tests...
2,fedimint_async,Async Wait,UNCERTAIN,A test is reported as flaky because fedimint-c...,The test exhibits flaky behavior under this un...,NOT_PRESENT,NOT_PRESENT
3,ethersync,Concurrency,E2,A fuzzer causes ungraceful shutdown of Tokio t...,CI fails intermittently.,Ungraceful shutdown of Tokio tasks is explicit...,NOT_PRESENT
4,vibranium,Logic,E3,The algorithm for merging default CLI options ...,The merging behavior produces incorrect/flaky ...,NOT_PRESENT,"The algorithm relies on sorting, which is iden..."
5,solana_async,Async Wait,E3/E4a,test_banking_stage_entryfication is reported a...,The test is being ignored because of the flaky...,NOT_PRESENT,The issue identifies a race condition exposed ...
6,deltachat,Network,E0,SMTP-related tests are reported to time out un...,Tests time out in CI.,"During high load, Postfix can return temporary...",src/smtp.rs::send_smtp_messages logs and ignor...
7,fedimint_logic,Logic,E0,A test reports an error involving the received...,Received: 0.000005 BTC; expected: 0.00000500 B...,The error occurs when running scripts/cli-test...,NOT_PRESENT
8,parsec-cloud,Hard to classify,E4b,A hypothesis test failed during a GitHub Actio...,The issue only states that the hypothesis test...,NOT_PRESENT,NOT_PRESENT
9,relay,Time,E0,The start_time_from_timestamp test is reported...,"Assertion failed: left = 9, right = 10.",The now timestamp and system_time timestamp ca...,The affected test is extractors::start_time::t...


## 3. Cumulative evidence packages

The main ablation compares the same 15 cases under progressively richer evidence:

| Package | Evidence |
|---|---|
| **P0** | E0 |
| **P1** | E0 + E1 |
| **P2** | E0 + E1 + E2 |
| **P3** | E0 + E1 + E2 + E3 |

This directly tests whether adding another evidence source changes automated diagnosis.

In [7]:
# ============================================================
# BUILD CUMULATIVE EVIDENCE PACKAGES
# ============================================================

def valid_evidence(text):
    return (
        pd.notna(text)
        and str(text).strip() != ""
        and str(text).strip() != "NOT_PRESENT"
    )


def combine_evidence(row, levels):
    """
    Combine selected evidence levels into one text package.
    """
    parts = []

    for level in levels:
        text = row[level]

        if valid_evidence(text):
            parts.append(f"[{level}]\n{text}")

    if not parts:
        return "NO_EVIDENCE"

    return "\n\n".join(parts)


# Sequential core evidence
evidence_clean["P0_E0"] = evidence_clean.apply(
    lambda r: combine_evidence(r, ["E0_issue"]),
    axis=1
)

evidence_clean["P1_E0_E1"] = evidence_clean.apply(
    lambda r: combine_evidence(r, ["E0_issue", "E1_failure"]),
    axis=1
)

evidence_clean["P2_E0_E1_E2"] = evidence_clean.apply(
    lambda r: combine_evidence(r, ["E0_issue", "E1_failure", "E2_runtime"]),
    axis=1
)

evidence_clean["P3_E0_E1_E2_E3"] = evidence_clean.apply(
    lambda r: combine_evidence(
        r,
        ["E0_issue", "E1_failure", "E2_runtime", "E3_source"]
    ),
    axis=1
)

# External evidence branches
evidence_clean["P4a_CI"] = evidence_clean.apply(
    lambda r: combine_evidence(
        r,
        [
            "E0_issue",
            "E1_failure",
            "E2_runtime",
            "E3_source",
            "E4a_CI"
        ]
    ),
    axis=1
)

evidence_clean["P4b_attachment"] = evidence_clean.apply(
    lambda r: combine_evidence(
        r,
        [
            "E0_issue",
            "E1_failure",
            "E2_runtime",
            "E3_source",
            "E4b_attachment"
        ]
    ),
    axis=1
)

evidence_clean["P4c_PR"] = evidence_clean.apply(
    lambda r: combine_evidence(
        r,
        [
            "E0_issue",
            "E1_failure",
            "E2_runtime",
            "E3_source",
            "E4c_PR"
        ]
    ),
    axis=1
)


# Display the packages for inspection
display(
    evidence_clean[
        [
            "case",
            "ground_truth",
            "minimum_evidence",
            "P0_E0",
            "P1_E0_E1",
            "P2_E0_E1_E2",
            "P3_E0_E1_E2_E3"
        ]
    ]
)

,case,ground_truth,minimum_evidence,P0_E0,P1_E0_E1,P2_E0_E1_E2,P3_E0_E1_E2_E3
0,nearcore,Randomness,E4c,[E0_issue]\nA flaky test named test_trie_consi...,[E0_issue]\nA flaky test named test_trie_consi...,[E0_issue]\nA flaky test named test_trie_consi...,[E0_issue]\nA flaky test named test_trie_consi...
1,reth,Network,E2,[E0_issue]\nThe test test_incoming_node_id_bla...,[E0_issue]\nThe test test_incoming_node_id_bla...,[E0_issue]\nThe test test_incoming_node_id_bla...,[E0_issue]\nThe test test_incoming_node_id_bla...
2,fedimint_async,Async Wait,UNCERTAIN,[E0_issue]\nA test is reported as flaky becaus...,[E0_issue]\nA test is reported as flaky becaus...,[E0_issue]\nA test is reported as flaky becaus...,[E0_issue]\nA test is reported as flaky becaus...
3,ethersync,Concurrency,E2,[E0_issue]\nA fuzzer causes ungraceful shutdow...,[E0_issue]\nA fuzzer causes ungraceful shutdow...,[E0_issue]\nA fuzzer causes ungraceful shutdow...,[E0_issue]\nA fuzzer causes ungraceful shutdow...
4,vibranium,Logic,E3,[E0_issue]\nThe algorithm for merging default ...,[E0_issue]\nThe algorithm for merging default ...,[E0_issue]\nThe algorithm for merging default ...,[E0_issue]\nThe algorithm for merging default ...
5,solana_async,Async Wait,E3/E4a,[E0_issue]\ntest_banking_stage_entryfication i...,[E0_issue]\ntest_banking_stage_entryfication i...,[E0_issue]\ntest_banking_stage_entryfication i...,[E0_issue]\ntest_banking_stage_entryfication i...
6,deltachat,Network,E0,[E0_issue]\nSMTP-related tests are reported to...,[E0_issue]\nSMTP-related tests are reported to...,[E0_issue]\nSMTP-related tests are reported to...,[E0_issue]\nSMTP-related tests are reported to...
7,fedimint_logic,Logic,E0,[E0_issue]\nA test reports an error involving ...,[E0_issue]\nA test reports an error involving ...,[E0_issue]\nA test reports an error involving ...,[E0_issue]\nA test reports an error involving ...
8,parsec-cloud,Hard to classify,E4b,[E0_issue]\nA hypothesis test failed during a ...,[E0_issue]\nA hypothesis test failed during a ...,[E0_issue]\nA hypothesis test failed during a ...,[E0_issue]\nA hypothesis test failed during a ...
9,relay,Time,E0,[E0_issue]\nThe start_time_from_timestamp test...,[E0_issue]\nThe start_time_from_timestamp test...,[E0_issue]\nThe start_time_from_timestamp test...,[E0_issue]\nThe start_time_from_timestamp test...


In [8]:
# Check how much evidence exists at each level

availability = pd.DataFrame({
    "Evidence package": [
        "E0",
        "E0 + E1",
        "E0 + E1 + E2",
        "E0 + E1 + E2 + E3",
        "E4a CI",
        "E4b attachment",
        "E4c PR"
    ],

    "Cases with evidence": [
        evidence_clean["E0_issue"].apply(valid_evidence).sum(),
        evidence_clean[["E0_issue", "E1_failure"]]
            .apply(lambda x: all(valid_evidence(v) for v in x), axis=1).sum(),
        evidence_clean[["E0_issue", "E1_failure", "E2_runtime"]]
            .apply(lambda x: all(valid_evidence(v) for v in x), axis=1).sum(),
        evidence_clean[["E0_issue", "E1_failure", "E2_runtime", "E3_source"]]
            .apply(lambda x: all(valid_evidence(v) for v in x), axis=1).sum(),
        evidence_clean["E4a_CI"].apply(valid_evidence).sum(),
        evidence_clean["E4b_attachment"].apply(valid_evidence).sum(),
        evidence_clean["E4c_PR"].apply(valid_evidence).sum()
    ]
})

display(availability)

,Evidence package,Cases with evidence
0,E0,15
1,E0 + E1,15
2,E0 + E1 + E2,7
3,E0 + E1 + E2 + E3,5
4,E4a CI,9
5,E4b attachment,1
6,E4c PR,5


## 4. Automated baseline: BART zero-shot classification

`facebook/bart-large-mnli` is used only as a **simple diagnostic baseline**. It is not treated as a proxy for the capabilities of current LLMs.

The model predicts one of the nine RustFT root-cause categories for each evidence package.

In [9]:
# ============================================================
# BART ZERO-SHOT DIAGNOSIS
# Evidence Accumulation Experiment
# ============================================================

from transformers import pipeline
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Load model
# ------------------------------------------------------------

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

# ------------------------------------------------------------
# 2. Root-cause categories
# ------------------------------------------------------------

labels = [
    "Randomness",
    "Network",
    "Async Wait",
    "Concurrency",
    "Logic",
    "Hard to classify",
    "Time",
    "I/O",
    "Unordered data"
]

hypotheses = {
    "Randomness":
        "This flaky test failure is caused by randomness or random input.",

    "Network":
        "This flaky test failure is caused by a network communication problem.",

    "Async Wait":
        "This flaky test failure is caused by insufficient waiting for an asynchronous operation.",

    "Concurrency":
        "This flaky test failure is caused by concurrent execution, a race condition, or task interleaving.",

    "Logic":
        "This flaky test failure is caused by incorrect program logic or an algorithmic problem.",

    "Hard to classify":
        "The available evidence is insufficient to determine the root cause.",

    "Time":
        "This flaky test failure is caused by time, timestamps, clocks, or timing boundaries.",

    "I/O":
        "This flaky test failure is caused by file, storage, filesystem, input, or output operations.",

    "Unordered data":
        "This flaky test failure is caused by nondeterministic ordering of data or output."
}

candidate_hypotheses = [
    hypotheses[label] for label in labels
]

# ------------------------------------------------------------
# 3. Function for one prediction
# ------------------------------------------------------------

def bart_predict(text):

    if text is None or str(text).strip() == "" or text == "NO_EVIDENCE":
        return "UNKNOWN", 0.0

    result = classifier(
        text,
        candidate_hypotheses,
        multi_label=False
    )

    best_hypothesis = result["labels"][0]
    score = result["scores"][0]

    # map hypothesis back to category
    prediction = labels[
        candidate_hypotheses.index(best_hypothesis)
    ]

    return prediction, score


# ------------------------------------------------------------
# 4. Evidence packages to evaluate
# ------------------------------------------------------------

packages = {
    "P0_E0": "P0_E0",
    "P1_E0_E1": "P1_E0_E1",
    "P2_E0_E1_E2": "P2_E0_E1_E2",
    "P3_E0_E1_E2_E3": "P3_E0_E1_E2_E3"
}

# ------------------------------------------------------------
# 5. Run experiment
# ------------------------------------------------------------

results = []

for package_name, package_column in packages.items():

    print(f"\nRunning {package_name} ...")

    for idx, row in tqdm(
        evidence_clean.iterrows(),
        total=len(evidence_clean)
    ):

        text = row[package_column]

        prediction, confidence = bart_predict(text)

        results.append({
            "case": row["case"],
            "ground_truth": row["ground_truth"],
            "package": package_name,
            "prediction": prediction,
            "confidence": confidence,
            "minimum_evidence": row["minimum_evidence"]
        })


results_df = pd.DataFrame(results)

print("\nExperiment complete.")

display(results_df)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

  0%|          | 0/15 [00:00<?, ?it/s]


Running P3_E0_E1_E2_E3 ...


  0%|          | 0/15 [00:00<?, ?it/s]


Experiment complete.


,case,ground_truth,package,prediction,confidence,minimum_evidence
0,nearcore,Randomness,P0_E0,Randomness,0.400888,E4c
1,reth,Network,P0_E0,Unordered data,0.350564,E2
2,fedimint_async,Async Wait,P0_E0,Unordered data,0.452065,UNCERTAIN
3,ethersync,Concurrency,P0_E0,Unordered data,0.524056,E2
4,vibranium,Logic,P0_E0,Logic,0.643223,E3
5,solana_async,Async Wait,P0_E0,Hard to classify,0.331842,E3/E4a
6,deltachat,Network,P0_E0,Time,0.189350,E0
7,fedimint_logic,Logic,P0_E0,Unordered data,0.385116,E0
8,parsec-cloud,Hard to classify,P0_E0,Hard to classify,0.776300,E4b
9,relay,Time,P0_E0,Unordered data,0.272533,E0


In [10]:
from sklearn.metrics import accuracy_score, f1_score

metric_rows = []

for package in ["P0_E0", "P1_E0_E1", "P2_E0_E1_E2", "P3_E0_E1_E2_E3"]:

    subset = results_df[results_df["package"] == package]

    y_true = subset["ground_truth"]
    y_pred = subset["prediction"]

    metric_rows.append({
        "package": package,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_F1": f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0
        ),
        "weighted_F1": f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="weighted",
            zero_division=0
        )
    })

metrics_df = pd.DataFrame(metric_rows)

metrics_df["accuracy"] *= 100
metrics_df["macro_F1"] *= 100
metrics_df["weighted_F1"] *= 100

metrics_df.round(2)

,package,accuracy,macro_F1,weighted_F1
0,P0_E0,20.0,18.52,20.00
1,P1_E0_E1,20.0,13.05,12.28
2,P2_E0_E1_E2,40.0,40.21,37.46
3,P3_E0_E1_E2_E3,40.0,40.21,37.46


### Preliminary automated result

In this 15-case pilot:

- P0 (issue description): **20% accuracy**
- P1 (+ failure evidence): **20%**
- P2 (+ runtime evidence): **40%**
- P3 (+ source/execution-path evidence): **40%**

The largest improvement occurred when runtime evidence was introduced. This is a **preliminary signal**, not a statistically established conclusion.

## 5. External artifacts: separate analysis

External evidence is analyzed separately because a referenced artifact is not equivalent to retrieved evidence. During the pilot, only a very small number of cases had genuinely retrieved external artifacts.

This analysis is therefore treated as a methodological check rather than a basis for a substantive conclusion.

In [11]:
# ============================================================
# ACTUAL EXTERNAL ARTIFACT EVIDENCE
# ============================================================

external_evidence = pd.DataFrame([
    {
        "case": "nearcore",
        "artifact_type": "E4c_PR",
        "artifact_status": "CONTENT_RETRIEVED",
        "diagnostic_value": "LIMITED",
        "artifact_evidence": (
            "Issue #10089 states that PR #10015 introduced "
            "test_trie_consistency_random, which fails about 2% "
            "of the time. The PR implements MemTrie update logic "
            "and includes randomized testing."
        )
    },

    {
        "case": "reth",
        "artifact_type": "E4a_CI",
        "artifact_status": "REFERENCED_ONLY",
        "diagnostic_value": "UNAVAILABLE",
        "artifact_evidence": ""
    },

    {
        "case": "solana_async",
        "artifact_type": "E4a_CI",
        "artifact_status": "REFERENCED_ONLY",
        "diagnostic_value": "UNAVAILABLE",
        "artifact_evidence": ""
    },

    {
        "case": "parsec-cloud",
        "artifact_type": "E4b_attachment",
        "artifact_status": "REFERENCED_ONLY",
        "diagnostic_value": "UNAVAILABLE",
        "artifact_evidence": ""
    },

    {
        "case": "solana_randomness",
        "artifact_type": "E4a_CI",
        "artifact_status": "REFERENCED_ONLY",
        "diagnostic_value": "UNAVAILABLE",
        "artifact_evidence": ""
    },

    {
        "case": "diem",
        "artifact_type": "E4a_CI",
        "artifact_status": "REFERENCED_ONLY",
        "diagnostic_value": "UNAVAILABLE",
        "artifact_evidence": ""
    },

    {
        "case": "webrender",
        "artifact_type": "E4c_PR",
        "artifact_status": "CONTENT_RETRIEVED",
        "diagnostic_value": "HIGH",
        "artifact_evidence": (
            "The commit removes a hard-coded max_texture_layers "
            "value of 2048 and changes max_texture_layers() to "
            "return self.limits.max_image_array_layers."
        )
    }
])

display(external_evidence)

,case,artifact_type,artifact_status,diagnostic_value,artifact_evidence
0,nearcore,E4c_PR,CONTENT_RETRIEVED,LIMITED,Issue #10089 states that PR #10015 introduced ...
1,reth,E4a_CI,REFERENCED_ONLY,UNAVAILABLE,
2,solana_async,E4a_CI,REFERENCED_ONLY,UNAVAILABLE,
3,parsec-cloud,E4b_attachment,REFERENCED_ONLY,UNAVAILABLE,
4,solana_randomness,E4a_CI,REFERENCED_ONLY,UNAVAILABLE,
5,diem,E4a_CI,REFERENCED_ONLY,UNAVAILABLE,
6,webrender,E4c_PR,CONTENT_RETRIEVED,HIGH,The commit removes a hard-coded max_texture_la...


In [12]:
# ============================================================
# CREATE ONLY GENUINE EXTERNAL-EVIDENCE PACKAGES
# ============================================================

e4_map = dict(
    zip(
        external_evidence["case"],
        external_evidence["artifact_evidence"]
    )
)

evidence_clean["E4_actual"] = evidence_clean["case"].map(e4_map)

def make_actual_e4_package(row):

    base = row["P3_E0_E1_E2_E3"]

    external = row["E4_actual"]

    if pd.isna(external) or str(external).strip() == "":
        return None

    return base + "\n\n[E4_external_artifact]\n" + external


evidence_clean["P4_actual"] = evidence_clean.apply(
    make_actual_e4_package,
    axis=1
)

actual_cases = evidence_clean[
    evidence_clean["P4_actual"].notna()
][
    [
        "case",
        "ground_truth",
        "minimum_evidence",
        "P3_E0_E1_E2_E3",
        "E4_actual",
        "P4_actual"
    ]
]

display(actual_cases)


,case,ground_truth,minimum_evidence,P3_E0_E1_E2_E3,E4_actual,P4_actual
0,nearcore,Randomness,E4c,[E0_issue]\nA flaky test named test_trie_consi...,Issue #10089 states that PR #10015 introduced ...,[E0_issue]\nA flaky test named test_trie_consi...
14,webrender,I/O,E3,[E0_issue]\nTwo WebRender tests are reported a...,The commit removes a hard-coded max_texture_la...,[E0_issue]\nTwo WebRender tests are reported a...


In [13]:
# ============================================================
# P3 vs ACTUAL E4 — PAIRED EXPERIMENT
# ============================================================

paired = evidence_clean[
    evidence_clean["P4_actual"].notna()
].copy()

print("Number of paired cases:", len(paired))
print("\nCases:")
print(paired[["case", "ground_truth", "minimum_evidence"]].to_string(index=False))

Number of paired cases: 2

Cases:
     case ground_truth minimum_evidence
 nearcore   Randomness              E4c
webrender          I/O               E3


In [14]:
# ============================================================
# BART: P3 vs P3 + ACTUAL EXTERNAL ARTIFACT
# ============================================================

from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0
)

labels = [
    "Randomness",
    "Network",
    "Async Wait",
    "Concurrency",
    "Logic",
    "Hard to classify",
    "Time",
    "I/O",
    "Unordered data"
]

hypotheses = {
    "Randomness":
        "This flaky test failure is caused by randomness or random input.",
    "Network":
        "This flaky test failure is caused by a network communication problem.",
    "Async Wait":
        "This flaky test failure is caused by insufficient waiting for an asynchronous operation.",
    "Concurrency":
        "This flaky test failure is caused by concurrent execution, a race condition, or task interleaving.",
    "Logic":
        "This flaky test failure is caused by incorrect program logic or an algorithmic problem.",
    "Hard to classify":
        "The available evidence is insufficient to determine the root cause.",
    "Time":
        "This flaky test failure is caused by time, timestamps, clocks, or timing boundaries.",
    "I/O":
        "This flaky test failure is caused by file, storage, filesystem, input, or output operations.",
    "Unordered data":
        "This flaky test failure is caused by nondeterministic ordering of data or output."
}

hypothesis_list = [hypotheses[x] for x in labels]


def predict_case(text):
    result = classifier(
        text,
        candidate_labels=hypothesis_list,
        multi_label=False
    )

    hypothesis_to_label = {
        hypotheses[label]: label
        for label in labels
    }

    return hypothesis_to_label[result["labels"][0]]


results_e4 = []

for _, row in paired.iterrows():

    pred_p3 = predict_case(row["P3_E0_E1_E2_E3"])
    pred_p4 = predict_case(row["P4_actual"])

    results_e4.append({
        "case": row["case"],
        "true": row["ground_truth"],
        "P3_prediction": pred_p3,
        "P4_prediction": pred_p4,
        "P3_correct": pred_p3 == row["ground_truth"],
        "P4_correct": pred_p4 == row["ground_truth"]
    })

e4_results = pd.DataFrame(results_e4)

display(e4_results)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

,case,true,P3_prediction,P4_prediction,P3_correct,P4_correct
0,nearcore,Randomness,Hard to classify,Hard to classify,False,False
1,webrender,I/O,I/O,Hard to classify,True,False


In [15]:
# ============================================================
# PAIRED E4 SUMMARY
# ============================================================

print("P3 accuracy:",
      e4_results["P3_correct"].mean())

print("P3 + actual E4 accuracy:",
      e4_results["P4_correct"].mean())

print("\nPrediction changes:")

changed = e4_results[
    e4_results["P3_prediction"] != e4_results["P4_prediction"]
]

display(changed)

P3 accuracy: 0.5
P3 + actual E4 accuracy: 0.0

Prediction changes:


,case,true,P3_prediction,P4_prediction,P3_correct,P4_correct
1,webrender,I/O,I/O,Hard to classify,True,False


## 6. Human minimum-sufficient-evidence annotation

The automated ablation does not answer the central question of **what evidence is sufficient for a knowledgeable engineer**. Therefore, each case is independently annotated with:

- minimum evidence level
- diagnosis status: `SUFFICIENT`, `AMBIGUOUS`, or `UNAVAILABLE`
- confidence
- diagnostic reason

The conservative rule is important: if the available material does not justify a diagnosis, the case is not forced into a minimum evidence label.

In [16]:
# ============================================================
# MINIMUM SUFFICIENT EVIDENCE — PILOT ANNOTATION
# ============================================================

minimum_evidence_final = pd.DataFrame([
    {
        "case": "nearcore",
        "ground_truth": "Randomness",
        "minimum_evidence": "E4c",
        "confidence": "Low",
        "reason": "The issue reports intermittent failure and links the PR that introduced the randomized test, but the artifact does not directly establish the causal mechanism."
    },
    {
        "case": "reth",
        "ground_truth": "Network",
        "minimum_evidence": "E2",
        "confidence": "High",
        "reason": "The HTTPError explicitly shows TCP connection refusal to localhost."
    },
    {
        "case": "fedimint_async",
        "ground_truth": "Async Wait",
        "minimum_evidence": "E0",
        "confidence": "Medium",
        "reason": "The issue directly attributes flaky behavior to asynchronous spend behavior, but the exact synchronization mechanism is not shown."
    },
    {
        "case": "ethersync",
        "ground_truth": "Concurrency",
        "minimum_evidence": "E2",
        "confidence": "High",
        "reason": "The issue explicitly identifies ungraceful shutdown of Tokio tasks as the source of intermittent CI failure."
    },
    {
        "case": "vibranium",
        "ground_truth": "Logic",
        "minimum_evidence": "E3",
        "confidence": "High",
        "reason": "The source-level explanation identifies sorting as an inappropriate algorithmic mechanism."
    },
    {
        "case": "solana_async",
        "ground_truth": "Async Wait",
        "minimum_evidence": "E3",
        "confidence": "Medium",
        "reason": "The issue explicitly identifies a race condition in the affected test, although the complete runtime evidence is unavailable."
    },
    {
        "case": "deltachat",
        "ground_truth": "Network",
        "minimum_evidence": "E2",
        "confidence": "High",
        "reason": "The temporary SMTP 421 timeout under load directly identifies the network/service failure mechanism."
    },
    {
        "case": "fedimint_logic",
        "ground_truth": "Logic",
        "minimum_evidence": "E1",
        "confidence": "High",
        "reason": "The expected and received values are numerically identical but represented differently, revealing the comparison/representation problem."
    },
    {
        "case": "parsec-cloud",
        "ground_truth": "Hard to classify",
        "minimum_evidence": "E4b",
        "confidence": "Low",
        "reason": "The issue only identifies a failed hypothesis test and points to an unavailable diagnostic ZIP; the available text itself is insufficient."
    },
    {
        "case": "relay",
        "ground_truth": "Time",
        "minimum_evidence": "E2",
        "confidence": "High",
        "reason": "The issue explicitly explains that the two timestamp calls can cross a one-second boundary."
    },
    {
        "case": "solana_randomness",
        "ground_truth": "Randomness",
        "minimum_evidence": "E4a",
        "confidence": "Low",
        "reason": "The issue identifies CI failures and the affected tests, but the actual CI diagnostic output is unavailable."
    },
    {
        "case": "diem",
        "ground_truth": "I/O",
        "minimum_evidence": "E4a",
        "confidence": "Low",
        "reason": "The issue identifies a storage test and links CI failures, but the actual failure output is unavailable."
    },
    {
        "case": "rust-lightning",
        "ground_truth": "Concurrency",
        "minimum_evidence": "E2",
        "confidence": "Medium",
        "reason": "The first read_event unexpectedly errors during execution, but the complete concurrency mechanism is not shown."
    },
    {
        "case": "databend",
        "ground_truth": "Unordered data",
        "minimum_evidence": "E1",
        "confidence": "High",
        "reason": "The output differs only in the ordering of timezone values."
    },
    {
        "case": "webrender",
        "ground_truth": "I/O",
        "minimum_evidence": "E2",
        "confidence": "High",
        "reason": "The issue explicitly states that the crash occurs because the process runs out of file descriptors."
    }
])

display(minimum_evidence_final)

,case,ground_truth,minimum_evidence,confidence,reason
0,nearcore,Randomness,E4c,Low,The issue reports intermittent failure and lin...
1,reth,Network,E2,High,The HTTPError explicitly shows TCP connection ...
2,fedimint_async,Async Wait,E0,Medium,The issue directly attributes flaky behavior t...
3,ethersync,Concurrency,E2,High,The issue explicitly identifies ungraceful shu...
4,vibranium,Logic,E3,High,The source-level explanation identifies sortin...
5,solana_async,Async Wait,E3,Medium,The issue explicitly identifies a race conditi...
6,deltachat,Network,E2,High,The temporary SMTP 421 timeout under load dire...
7,fedimint_logic,Logic,E1,High,The expected and received values are numerical...
8,parsec-cloud,Hard to classify,E4b,Low,The issue only identifies a failed hypothesis ...
9,relay,Time,E2,High,The issue explicitly explains that the two tim...


In [17]:
# ============================================================
# MINIMUM EVIDENCE DISTRIBUTION
# ============================================================

distribution = (
    minimum_evidence_final
    .groupby("minimum_evidence")
    .size()
    .reset_index(name="cases")
)

distribution["percentage"] = (
    distribution["cases"] /
    len(minimum_evidence_final) * 100
).round(2)

display(distribution)

,minimum_evidence,cases,percentage
0,E0,1,6.67
1,E1,2,13.33
2,E2,6,40.00
3,E3,2,13.33
4,E4a,2,13.33
5,E4b,1,6.67
6,E4c,1,6.67


In [18]:
confidence_distribution = (
    minimum_evidence_final
    .groupby("confidence")
    .size()
    .reset_index(name="cases")
)

confidence_distribution["percentage"] = (
    confidence_distribution["cases"] /
    len(minimum_evidence_final) * 100
).round(2)

display(confidence_distribution)

,confidence,cases,percentage
0,High,8,53.33
1,Low,4,26.67
2,Medium,3,20.00


In [19]:
# ============================================================
# ISSUE-ONLY DIAGNOSABILITY
# ============================================================

issue_only = minimum_evidence_final[
    minimum_evidence_final["minimum_evidence"] == "E0"
]

print(
    f"Issue-only diagnosable: "
    f"{len(issue_only)}/{len(minimum_evidence_final)} "
    f"({len(issue_only)/len(minimum_evidence_final)*100:.1f}%)"
)

print("\nCases diagnosable from E0:")
display(issue_only[["case", "ground_truth", "reason"]])

Issue-only diagnosable: 1/15 (6.7%)

Cases diagnosable from E0:


,case,ground_truth,reason
2,fedimint_async,Async Wait,The issue directly attributes flaky behavior t...


### Minimum-evidence pilot result

Of the 15 cases:

- **8/15** were confidently sufficient
- **4/15** were ambiguous
- **3/15** were unavailable

Among the 8 confidently sufficient cases, runtime evidence (**E2**) was the minimum sufficient evidence in **5 cases (62.5%)**.

This percentage applies only to the confidently annotated pilot subset; it must not be generalized to the broader flaky-test population.

In [20]:
# ============================================================
# CUMULATIVE EVIDENCE REQUIREMENT
# ============================================================

level_order = {
    "E0": 0,
    "E1": 1,
    "E2": 2,
    "E3": 3,
    "E4a": 4,
    "E4b": 4,
    "E4c": 4
}

minimum_evidence_final["level"] = (
    minimum_evidence_final["minimum_evidence"]
    .map(level_order)
)

minimum_evidence_final["evidence_stage"] = (
    minimum_evidence_final["level"]
    .map({
        0: "Issue description",
        1: "Failure evidence",
        2: "Runtime evidence",
        3: "Source/execution-path evidence",
        4: "External artifact"
    })
)

stage_distribution = (
    minimum_evidence_final
    .groupby(["level", "evidence_stage"])
    .size()
    .reset_index(name="cases")
    .sort_values("level")
)

stage_distribution["percentage"] = (
    stage_distribution["cases"] /
    len(minimum_evidence_final) * 100
).round(2)

display(stage_distribution)

,level,evidence_stage,cases,percentage
0,0,Issue description,1,6.67
1,1,Failure evidence,2,13.33
2,2,Runtime evidence,6,40.00
3,3,Source/execution-path evidence,2,13.33
4,4,External artifact,4,26.67


In [21]:
# ============================================================
# EVIDENCE SUFFICIENCY MATRIX
# ============================================================

evidence_matrix = minimum_evidence_final[
    [
        "case",
        "ground_truth",
        "minimum_evidence",
        "confidence",
        "reason"
    ]
].copy()

evidence_matrix["E0"] = (
    evidence_matrix["minimum_evidence"] == "E0"
)

evidence_matrix["E1"] = (
    evidence_matrix["minimum_evidence"].isin(["E0", "E1"])
)

evidence_matrix["E2"] = (
    evidence_matrix["minimum_evidence"].isin(["E0", "E1", "E2"])
)

evidence_matrix["E3"] = (
    evidence_matrix["minimum_evidence"].isin(
        ["E0", "E1", "E2", "E3"]
    )
)

evidence_matrix["External"] = (
    evidence_matrix["minimum_evidence"].isin(
        ["E4a", "E4b", "E4c"]
    )
)

display(
    evidence_matrix[
        [
            "case",
            "ground_truth",
            "minimum_evidence",
            "confidence",
            "E0",
            "E1",
            "E2",
            "E3",
            "External"
        ]
    ]
)

,case,ground_truth,minimum_evidence,confidence,E0,E1,E2,E3,External
0,nearcore,Randomness,E4c,Low,False,False,False,False,True
1,reth,Network,E2,High,False,False,True,True,False
2,fedimint_async,Async Wait,E0,Medium,True,True,True,True,False
3,ethersync,Concurrency,E2,High,False,False,True,True,False
4,vibranium,Logic,E3,High,False,False,False,True,False
5,solana_async,Async Wait,E3,Medium,False,False,False,True,False
6,deltachat,Network,E2,High,False,False,True,True,False
7,fedimint_logic,Logic,E1,High,False,True,True,True,False
8,parsec-cloud,Hard to classify,E4b,Low,False,False,False,False,True
9,relay,Time,E2,High,False,False,True,True,False


In [22]:
# ============================================================
# ROOT CAUSE × MINIMUM EVIDENCE
# ============================================================

root_cause_evidence = pd.crosstab(
    minimum_evidence_final["ground_truth"],
    minimum_evidence_final["minimum_evidence"]
)

display(root_cause_evidence)

minimum_evidence,E0,E1,E2,E3,E4a,E4b,E4c
ground_truth,,,,,,,
Async Wait,1,0,0,1,0,0,0
Concurrency,0,0,2,0,0,0,0
Hard to classify,0,0,0,0,0,1,0
I/O,0,0,1,0,1,0,0
Logic,0,1,0,1,0,0,0
Network,0,0,2,0,0,0,0
Randomness,0,0,0,0,1,0,1
Time,0,0,1,0,0,0,0
Unordered data,0,1,0,0,0,0,0


In [23]:
# ============================================================
# ROOT CAUSE × MINIMUM EVIDENCE ASSOCIATION
# Exploratory only — n=15
# ============================================================

from scipy.stats import chi2_contingency

ct = pd.crosstab(
    minimum_evidence_final["ground_truth"],
    minimum_evidence_final["minimum_evidence"]
)

chi2, p, dof, expected = chi2_contingency(ct)

n = ct.to_numpy().sum()
r, k = ct.shape

cramers_v = (
    (chi2 / n) /
    min(r - 1, k - 1)
) ** 0.5

print("Chi-square:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Cramer's V:", cramers_v)

print("\nIMPORTANT:")
print("This is exploratory only because n=15 and many expected cells are small.")

Chi-square: 55.000000000000014
p-value: 0.22671477824213138
Degrees of freedom: 48
Cramer's V: 0.7817359599705717

IMPORTANT:
This is exploratory only because n=15 and many expected cells are small.


In [24]:
# ============================================================
# HOW OFTEN IS E2 THE MINIMUM EVIDENCE?
# ============================================================

e2_dependence = (
    minimum_evidence_final
    .assign(
        requires_E2=lambda x:
        x["minimum_evidence"] == "E2"
    )
    .groupby("ground_truth")
    .agg(
        cases=("case", "count"),
        E2_minimum=("requires_E2", "sum")
    )
    .reset_index()
)

e2_dependence["E2_fraction"] = (
    e2_dependence["E2_minimum"] /
    e2_dependence["cases"]
).round(2)

display(e2_dependence)

,ground_truth,cases,E2_minimum,E2_fraction
0,Async Wait,2,0,0.0
1,Concurrency,2,2,1.0
2,Hard to classify,1,0,0.0
3,I/O,2,1,0.5
4,Logic,2,0,0.0
5,Network,2,2,1.0
6,Randomness,2,0,0.0
7,Time,1,1,1.0
8,Unordered data,1,0,0.0


## 7. Annotation protocol and reliability

Because minimum-sufficient-evidence annotation involves human judgment, the next validation step is a **blind second annotation pass** without exposing the first-pass decisions.

Agreement will be measured using:
- percentage agreement
- Cohen's κ

Ideally, a second independent annotator should also perform the annotation to establish inter-annotator reliability before the study is scaled.

In [25]:
# ============================================================
# FORMAL ANNOTATION SCHEMA
# ============================================================

annotation_columns = [
    "case",
    "ground_truth",

    # Evidence availability
    "E0_issue",
    "E1_failure",
    "E2_runtime",
    "E3_source",
    "E4a_CI",
    "E4b_attachment",
    "E4c_PR",

    # Artifact retrieval
    "E4a_retrieved",
    "E4b_retrieved",
    "E4c_retrieved",

    # Diagnosis
    "minimum_evidence",
    "diagnosis_status",
    "confidence",

    # Human reasoning
    "diagnostic_reason"
]

annotation_df = pd.DataFrame(
    columns=annotation_columns
)

print("Annotation columns:")
for i, c in enumerate(annotation_columns, 1):
    print(i, c)

print("\nTotal columns:", len(annotation_columns))


Annotation columns:
1 case
2 ground_truth
3 E0_issue
4 E1_failure
5 E2_runtime
6 E3_source
7 E4a_CI
8 E4b_attachment
9 E4c_PR
10 E4a_retrieved
11 E4b_retrieved
12 E4c_retrieved
13 minimum_evidence
14 diagnosis_status
15 confidence
16 diagnostic_reason

Total columns: 16


In [26]:
# ============================================================
# INITIALIZE ANNOTATION TABLE FROM EXISTING PILOT
# ============================================================

annotation_df = evidence_clean[
    [
        "case",
        "ground_truth",
        "E0_issue",
        "E1_failure",
        "E2_runtime",
        "E3_source",
        "E4a_CI",
        "E4b_attachment",
        "E4c_PR"
    ]
].copy()

# Artifact retrieval status
annotation_df["E4a_retrieved"] = "NO"
annotation_df["E4b_retrieved"] = "NO"
annotation_df["E4c_retrieved"] = "NO"

# Fill known retrievals
annotation_df.loc[
    annotation_df["case"] == "webrender",
    "E4c_retrieved"
] = "YES"

annotation_df.loc[
    annotation_df["case"] == "nearcore",
    "E4c_retrieved"
] = "YES"

# Diagnosis fields
annotation_df["minimum_evidence"] = ""
annotation_df["diagnosis_status"] = ""
annotation_df["confidence"] = ""
annotation_df["diagnostic_reason"] = ""

display(annotation_df)

,case,ground_truth,E0_issue,E1_failure,E2_runtime,E3_source,E4a_CI,E4b_attachment,E4c_PR,E4a_retrieved,E4b_retrieved,E4c_retrieved,minimum_evidence,diagnosis_status,confidence,diagnostic_reason
0,nearcore,Randomness,A flaky test named test_trie_consistency_rando...,The test is reported as failing intermittently.,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,A linked PR (near/nearcore/pull/10015) introdu...,NO,NO,YES,,,,
1,reth,Network,The test test_incoming_node_id_blacklist is re...,The test panicked because Result::unwrap() rec...,The Err is an HTTPError caused by a TCP connec...,The failure occurs at crates/net/network/tests...,The issue links to a GitHub Actions test run.,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
2,fedimint_async,Async Wait,A test is reported as flaky because fedimint-c...,The test exhibits flaky behavior under this un...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
3,ethersync,Concurrency,A fuzzer causes ungraceful shutdown of Tokio t...,CI fails intermittently.,Ungraceful shutdown of Tokio tasks is explicit...,NOT_PRESENT,The report states that the problem makes CI fa...,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
4,vibranium,Logic,The algorithm for merging default CLI options ...,The merging behavior produces incorrect/flaky ...,NOT_PRESENT,"The algorithm relies on sorting, which is iden...",NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
5,solana_async,Async Wait,test_banking_stage_entryfication is reported a...,The test is being ignored because of the flaky...,NOT_PRESENT,The issue identifies a race condition exposed ...,The issue references an external Buildkite tes...,NOT_PRESENT,The linked change ignores test_banking_stage_e...,NO,NO,NO,,,,
6,deltachat,Network,SMTP-related tests are reported to time out un...,Tests time out in CI.,"During high load, Postfix can return temporary...",src/smtp.rs::send_smtp_messages logs and ignor...,The issue states that tests time out in CI.,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
7,fedimint_logic,Logic,A test reports an error involving the received...,Received: 0.000005 BTC; expected: 0.00000500 B...,The error occurs when running scripts/cli-test...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,
8,parsec-cloud,Hard to classify,A hypothesis test failed during a GitHub Actio...,The issue only states that the hypothesis test...,NOT_PRESENT,NOT_PRESENT,The issue references GitHub Actions run 288791...,A diagnostic log trace is contained in macos-h...,NOT_PRESENT,NO,NO,NO,,,,
9,relay,Time,The start_time_from_timestamp test is reported...,"Assertion failed: left = 9, right = 10.",The now timestamp and system_time timestamp ca...,The affected test is extractors::start_time::t...,A GitHub Actions test-run link is provided.,NOT_PRESENT,NOT_PRESENT,NO,NO,NO,,,,


In [27]:
# ============================================================
# BLIND SECOND-PASS ANNOTATION
# ============================================================

blind_annotation = annotation_df[
    [
        "case",
        "ground_truth",
        "E0_issue",
        "E1_failure",
        "E2_runtime",
        "E3_source",
        "E4a_CI",
        "E4b_attachment",
        "E4c_PR"
    ]
].copy()

# Remove previous decisions
blind_annotation["minimum_evidence_pass2"] = ""
blind_annotation["diagnosis_status_pass2"] = ""
blind_annotation["confidence_pass2"] = ""
blind_annotation["reason_pass2"] = ""

display(blind_annotation)

,case,ground_truth,E0_issue,E1_failure,E2_runtime,E3_source,E4a_CI,E4b_attachment,E4c_PR,minimum_evidence_pass2,diagnosis_status_pass2,confidence_pass2,reason_pass2
0,nearcore,Randomness,A flaky test named test_trie_consistency_rando...,The test is reported as failing intermittently.,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,A linked PR (near/nearcore/pull/10015) introdu...,,,,
1,reth,Network,The test test_incoming_node_id_blacklist is re...,The test panicked because Result::unwrap() rec...,The Err is an HTTPError caused by a TCP connec...,The failure occurs at crates/net/network/tests...,The issue links to a GitHub Actions test run.,NOT_PRESENT,NOT_PRESENT,,,,
2,fedimint_async,Async Wait,A test is reported as flaky because fedimint-c...,The test exhibits flaky behavior under this un...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,,,,
3,ethersync,Concurrency,A fuzzer causes ungraceful shutdown of Tokio t...,CI fails intermittently.,Ungraceful shutdown of Tokio tasks is explicit...,NOT_PRESENT,The report states that the problem makes CI fa...,NOT_PRESENT,NOT_PRESENT,,,,
4,vibranium,Logic,The algorithm for merging default CLI options ...,The merging behavior produces incorrect/flaky ...,NOT_PRESENT,"The algorithm relies on sorting, which is iden...",NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,,,,
5,solana_async,Async Wait,test_banking_stage_entryfication is reported a...,The test is being ignored because of the flaky...,NOT_PRESENT,The issue identifies a race condition exposed ...,The issue references an external Buildkite tes...,NOT_PRESENT,The linked change ignores test_banking_stage_e...,,,,
6,deltachat,Network,SMTP-related tests are reported to time out un...,Tests time out in CI.,"During high load, Postfix can return temporary...",src/smtp.rs::send_smtp_messages logs and ignor...,The issue states that tests time out in CI.,NOT_PRESENT,NOT_PRESENT,,,,
7,fedimint_logic,Logic,A test reports an error involving the received...,Received: 0.000005 BTC; expected: 0.00000500 B...,The error occurs when running scripts/cli-test...,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,NOT_PRESENT,,,,
8,parsec-cloud,Hard to classify,A hypothesis test failed during a GitHub Actio...,The issue only states that the hypothesis test...,NOT_PRESENT,NOT_PRESENT,The issue references GitHub Actions run 288791...,A diagnostic log trace is contained in macos-h...,NOT_PRESENT,,,,
9,relay,Time,The start_time_from_timestamp test is reported...,"Assertion failed: left = 9, right = 10.",The now timestamp and system_time timestamp ca...,The affected test is extractors::start_time::t...,A GitHub Actions test-run link is provided.,NOT_PRESENT,NOT_PRESENT,,,,


In [28]:
# ============================================================
# SAVE PASS-1 DECISIONS SEPARATELY
# ============================================================

pass1 = annotation_df[
    [
        "case",
        "minimum_evidence",
        "diagnosis_status",
        "confidence"
    ]
].copy()

pass1.columns = [
    "case",
    "minimum_evidence_pass1",
    "diagnosis_status_pass1",
    "confidence_pass1"
]

display(pass1)

,case,minimum_evidence_pass1,diagnosis_status_pass1,confidence_pass1
0,nearcore,,,
1,reth,,,
2,fedimint_async,,,
3,ethersync,,,
4,vibranium,,,
5,solana_async,,,
6,deltachat,,,
7,fedimint_logic,,,
8,parsec-cloud,,,
9,relay,,,


## 8. Interpretation and research direction

### Preliminary observations

1. **Evidence matters:** the BART baseline improved from 20% to 40% when runtime evidence was added.
2. **Runtime evidence appears particularly informative:** E2 was the minimum sufficient evidence in 5/8 confidently diagnosed pilot cases.
3. **Evidence requirements may be root-cause dependent:** different cases reached defensible diagnoses at different evidence levels.

### Research hypotheses

**H1.** Different flaky-test root-cause categories require different evidence types for reliable diagnosis.

**H2.** Runtime evidence provides substantial additional diagnostic value for a subset of flaky-test root causes.

**H3.** The minimum sufficient evidence varies systematically with the root-cause category.

**H4.** Evidence beyond the minimum sufficient set provides diminishing diagnostic returns.

### Important distinction

The current pilot is still primarily **category-level diagnosis**. A stronger future system should move toward:

> **root-cause category → causal mechanism → responsible artifact/location → supporting evidence → grounded explanation**

The goal is therefore not simply to predict “Concurrency” or “Network,” but to produce a defensible causal diagnosis supported by the supplied evidence.

## 9. Explainability: proposed grounding evaluation

Explanation quality should be evaluated separately from category accuracy.

A future explanation evaluation can check:

1. **Diagnosis correctness**
2. **Evidence attribution** — can the system identify the evidence supporting the diagnosis?
3. **Evidence completeness** — does it use the important evidence needed for the conclusion?
4. **Unsupported claims** — does it introduce causal claims not established by the supplied evidence?

An exploratory claim-level grounding score could be:

> **supported factual claims / total factual claims**

This is a proposed prototype measure and would require validation before being treated as a formal metric.

## 10. Limitations

- **Small pilot:** only 15 cases were analyzed.
- **Single baseline model:** BART is a simple zero-shot baseline, not a modern LLM benchmark.
- **Manual evidence segmentation:** evidence boundaries were manually curated.
- **Artifact availability:** some referenced CI logs/attachments could not be retrieved.
- **Annotation subjectivity:** minimum-sufficiency judgments require reliability testing.
- **Sparse category × evidence table:** statistical association tests are exploratory only.

These limitations motivate the next phase rather than weakening the research question itself.

## 11. Next experimental phase

1. Complete blind Pass-2 annotation.
2. Measure agreement and, ideally, inter-annotator reliability.
3. Expand beyond the 15-case pilot.
4. Run controlled evidence ablations on the larger labeled set.
5. Analyze evidence contribution by root-cause category.
6. Move from category prediction to structured causal diagnosis.
7. Evaluate explanation grounding at the claim/evidence level.
8. Compare an LLM, program-analysis signals, and a hybrid approach **only after** the evidence requirements are empirically established.

## Current status

**Dataset:** RustFT  
**Pilot:** 15 cases  
**Root-cause categories:** 9  
**Baseline:** BART-large-MNLI  
**Evidence packages:** P0–P3  
**Pass-1 annotation:** complete  
**Confidently sufficient:** 8/15  
**Ambiguous:** 4/15  
**Unavailable:** 3/15  
**Next step:** blind Pass-2 annotation → agreement analysis → scale-up